# D5 · From a VP metadata record to a notebook

This notebook starts from one thing only, the IRI of a dataset record in the ERDERA Virtual Platform, and follows the metadata to the data. Every URL it touches comes out of the record. Nothing else is typed in.

The chain is portal row → dataset record (FAIR Data Point) → distribution record → data file → first analysis → provenance record.

Today the portal's resource list shows these datasets without a link to their record, so the IRI below has to be taken from the VP Index or from the FDP that published the dataset. That gap is part of what this demonstrator shows.

In [ ]:
# Parameters. Fill DATASET_IRI with the dcat:Dataset IRI of the record you want to follow,
# for example the EJPRD WP13 CAKUT proteome or the Huntington disease transcriptomics dataset.
DATASET_IRI = ""           # e.g. "https://<fdp-host>/dataset/<id>"
DOWNLOAD_DIR = "downloads"
OUTPUT_DIR = "output"
GROUP_COLUMN = None        # e.g. "group" to compare CAKUT vs control; None for a plain summary

assert DATASET_IRI, "Set DATASET_IRI to a dataset record IRI first."


In [ ]:
import sys
from datetime import datetime, timezone
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))   # the demonstrator folder, so that `src` imports
from src import vpfollow

started = datetime.now(timezone.utc)


## 1. The dataset record

Fetch the record with content negotiation (an FDP serves Turtle) and read what it says about itself.

In [ ]:
dataset, distributions = vpfollow.follow(DATASET_IRI)

print("Title      :", dataset.title)
print("Publisher  :", dataset.publisher_name or dataset.publisher_iri)
print("Licence    :", dataset.license)
print("Access     :", dataset.access_rights)
print("Themes     :", ", ".join(dataset.themes) or "(none)")
print("Keywords   :", ", ".join(dataset.keywords) or "(none)")
print("Landing    :", dataset.landing_page)
print("Distributions:", len(distributions))
print()
print(dataset.description)


## 2. The distributions

Each distribution is its own record. We only want the one we can download.

In [ ]:
import pandas as pd

pd.DataFrame([{
    "title": d.title,
    "media type": d.media_type,
    "download URL": d.download_url,
    "access URL": d.access_url,
    "licence": d.license,
} for d in distributions])


In [ ]:
chosen = vpfollow.choose(distributions)
assert chosen is not None, "No distribution carries a download or access URL; see the README fallback."
print("Following:", chosen.title)
print("URL      :", chosen.data_url)

downloaded = vpfollow.download(chosen.data_url, DOWNLOAD_DIR)
print("Saved to :", downloaded, f"({downloaded.stat().st_size:,} bytes)")


## 3. The data

Load the file and look at it. The analysis below is deliberately simple, the point is that we got here from the metadata alone.

In [ ]:
table = vpfollow.load_table(downloaded)
print(table.shape)
table.head()


In [ ]:
numeric = table.select_dtypes("number")

if GROUP_COLUMN and GROUP_COLUMN in table.columns:
    means = table.groupby(GROUP_COLUMN)[numeric.columns].mean().T
    groups = list(means.columns)
    if len(groups) == 2:
        means["difference"] = means[groups[0]] - means[groups[1]]
        means = means.sort_values("difference", key=abs, ascending=False)
    summary = means
else:
    summary = numeric.describe().T

summary.head(20)


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
if GROUP_COLUMN and GROUP_COLUMN in table.columns and "difference" in summary.columns:
    summary["difference"].head(20).plot.barh(ax=ax)
    ax.set_xlabel(f"mean difference ({groups[0]} − {groups[1]})")
    ax.set_title(dataset.title or DATASET_IRI)
else:
    numeric.iloc[:, : min(6, numeric.shape[1])].plot.box(ax=ax)
    ax.set_title(dataset.title or DATASET_IRI)
ax.invert_yaxis()
plt.tight_layout()


## 4. Provenance

A small PROV-O record of what this run fetched and produced, so the analysis itself is traceable.

In [ ]:
prov = vpfollow.prov_graph(dataset, chosen, downloaded, started)
Path(OUTPUT_DIR).mkdir(exist_ok=True)
prov_path = Path(OUTPUT_DIR) / "prov.ttl"
prov.serialize(destination=str(prov_path), format="turtle")
print(prov_path.read_text())


## What this shows

- A dataset in the VP can be reached and used from its metadata alone, with the licence and publisher known before any data is downloaded.
- The portal's resource list does not yet link to the record, so a user has to know the IRI. Closing that gap is a portal change worth requesting.
- The notebook itself can be registered as a resource in the demo index with FDPlight, which is the second act of this demonstrator.